<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

# 🤖 Lesson 0 · 本节精华

</div>

<small>对应主课:[0_a_🔥_create_agent.ipynb](./0_a_🔥_create_agent.ipynb)</small>

## 🎯 一句话定位

> 用 LangChain 的 **`create_agent`** 抽象搭最简 ReAct agent,搞懂 **graph / state / messages** 三件套,以及怎么让工具**安全访问 + 修改 state**。
>
> 🏗️ **这是后续所有 lesson 的地基**。

## 📚 你应该学到的 8 件事

| # | 概念 | 你应该记住的一句话 |
|---|---|---|
| 1 | 🧠 **ReAct 循环** | LLM ↔ 工具节点反复跳,**直到 LLM 不再产 tool_calls** → END |
| 2 | 🛠️ **`create_agent`** | 给它 4 样东西即可:`model` / `tools` / `system_prompt` / `state_schema` |
| 3 | 📦 **`AgentState` 默认结构** | `messages` 列表 + `add_messages` reducer(追加式) |
| 4 | 🔄 **Reducer 的意义** | 描述"新值如何并入旧值";并行写同一字段时尤其关键 |
| 5 | 🗂️ **自定义 State** | 继承 `AgentState`,新字段用 `Annotated[Type, your_reducer]` |
| 6 | 🪄 **`InjectedState`** | 让工具拿到 graph state,**但对 LLM 不可见**(框架自动注入) |
| 7 | 🪪 **`InjectedToolCallId`** | 同上,给工具用于**手动构造 `ToolMessage`** |
| 8 | 📤 **`Command(update={...})`** | 工具想**同时**更新 state 的多个字段时,用它取代普通 return |

## 🧩 必背心智模型 1:ReAct 循环 = 状态机

```
START → agent ──┬──► tools ──► agent   (无条件回边)
                └──► END                (条件边:AIMessage.tool_calls 为空就退出)
```

| 节点 / 边 | 做什么 |
|---|---|
| 🤖 **agent 节点** | 调用 LLM,产 1 条 `AIMessage` |
| 🔧 **tools 节点** | `ToolNode` 并发执行 AIMessage 里**所有** tool_calls |
| 🔀 **条件边** | 只看 `AIMessage.tool_calls` 是否为空 → 决定 `tools` or `END` |
| ↩️ **无条件回边** | tools → agent,总是回 LLM |

> 🔑 **关键事实**:**循环不在 LLM 里**,而在 LangGraph 这台状态机里。**LLM 是无状态的**,每轮重新接收完整 messages。

## 🧩 必背心智模型 2:State 注入 —— 怎么让工具看到 state

<div style="background:#ffe6e6; padding:10px 24px; border-left:4px solid #d9534f; border-radius:4px; width:97%;">

⚠️ **困境**:`calculator(operation, a, b, state)` 这种工具,LLM 该怎么填 `state`?**它根本看不到 state**。

</div>

**解法 = `InjectedState` 注解**(以及它兄弟 `InjectedToolCallId`):

<pre style="background:#f6f8fa; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">@tool
def calculator_wstate(
    operation, a, b,
    <span style="background:#fff3a3; padding:0 2px;">state: Annotated[CalcState, InjectedState],</span>          # ← <b>不发给 LLM</b>
    <span style="background:#fff3a3; padding:0 2px;">tool_call_id: Annotated[str, InjectedToolCallId],</span>   # ← <b>不发给 LLM</b>
):
    ...
</code></pre>

**机制**:
1. 🚫 这两个参数从 **发给 LLM 的 tool schema 里被剥掉**
2. 🪄 `ToolNode` 执行工具前,**框架自己把当前 state + tool_call_id 填进去**
3. ✅ LLM 只填 `operation / a / b`,框架补完剩下的

## 💎 关键代码模式:`Command` 同时更新多个字段

工具默认行为:`return result` → 框架自动把它包成 `ToolMessage` 加进 `messages`。

但如果想**同时更新别的字段**(比如自定义的 `ops` 列表),普通 return 不够,要用 `Command`:

```python
return Command(
    update={
        "ops":      ops,                                          # 更新自定义字段
        "messages": [ToolMessage(f"{result}", tool_call_id=tool_call_id)],  # 自己造 ToolMessage
    }
)
```

> 🔑 **副作用**:一旦你用 `Command`,框架**就不再自动包 ToolMessage 了** —— 你必须**自己造**,而且**必须传 `tool_call_id`**(这就是为什么前面要 `InjectedToolCallId`)。

这两个特性**互为前提**:`Command` 需要 ToolMessage,ToolMessage 需要 `tool_call_id`,`tool_call_id` 通过 `InjectedToolCallId` 进来。三位一体。

## 🚀 一个容易忽视的点:**并行 tool_calls**

回忆 `((15+23)×64+52)/6` 那个例子 —— LLM 在**一个 AIMessage** 里塞了 **4 个 tool_calls**,`ToolNode` 会**并发执行**它们。

```python
AIMessage(tool_calls=[
    {'name': 'calculator_wstate', 'args': {'operation': 'add',      'a': 15, 'b': 23}},
    {'name': 'calculator_wstate', 'args': {'operation': 'multiply', 'a': 64, 'b': 1}},   # 这里其实是 no-op
    {'name': 'calculator_wstate', 'args': {'operation': 'add',      'a': 52, 'b': 0}},
    {'name': 'calculator_wstate', 'args': {'operation': 'divide',   'a': 6,  'b': 1}},
])
```

这是 ReAct 的**性能利器** —— LLM 会自动识别**互不依赖**的工具并打包并行。后续 lesson 的子 agent 派单也用同一机制。

> 💡 **后果**:并行 tool 调用 = 多个工具同时返回 update → **必须有合适的 reducer 才能合并**,这就是为什么后面 Lesson 2 要定义 `file_reducer`。

## 🌍 跟真实产品的对应

| 你刚学的概念 | 真实产品里长这样 |
|---|---|
| 🧠 ReAct 循环 | **Claude Code / Cursor Agent / Devin** 的核心驱动 |
| 🪄 InjectedState | 工具拿到"会话级上下文"的标准做法,所有 agent 框架都需要 |
| 📤 Command 多字段更新 | 复杂 agent state 管理的事实标准(LangGraph 1.0 推荐) |
| 🚀 并行 tool_calls | Claude / GPT 都已成熟支持,生产 agent 普遍使用 |

## ✨ 一句话带走

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

> 🔑 **Agent ≠ 一个 LLM,Agent = 一台跑着 ReAct 循环的状态机**。
>
> LLM 只负责"想 + 写 tool_call",**框架(LangGraph)管循环、管 state、管工具执行**。`InjectedState` + `Command` 让工具能与 state 双向通信,而 **LLM 自始至终对 state 不可见** —— 它只看 messages。

</div>